## Title : Machine Learning Based **Cyber Intrusion Detection System**

**Problem Statement:** The Objective of this project is to build ML Model That **Detects the Cyber Intrusions from network traffic data**.

**Metric** : **Recall with Precision & F1 Score**

+ Recall is main metric for this Project
+ Because Detecting all Intrusions is more Important than Overall accuracy 

**Task Type** :  **Supervised Binary Classification**

The problem is formulated as a supervised binary classification task where the **model learns to distinguish between normal and attack network traffic.**

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

In [ ]:
# Display settings
pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8,5)

In [ ]:
df = pd.read_csv(r"C:\Users\dadof\Downloads\cybersecurity_intrusion_data (1).csv")
df

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

### Data Cleaning 

In [ ]:
# Drop session_id 
if "session_id" in df.columns:
    df = df.drop(columns=["session_id"])

# Fill missing categorical values safely
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown")

# Final missing check
print("Missing Values After Cleaning:\n")
print(df.isnull().sum())

# Save cleaned dataset
df.to_csv("cleaned_data.csv", index=False)

In [ ]:
num_cols = df.select_dtypes(include= ["int", "float"]).columns
num_cols

In [ ]:
cat_cols = df.select_dtypes(include="object").columns
cat_cols

### Univariate Analysis

**Numerical Features** 

In [ ]:
for col in num_cols:
    plt.figure()
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
for col in num_cols:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

**Categorical Features**

In [ ]:
for col in cat_cols:
    plt.figure()
    sns.countplot(x=df[col])
    plt.title(f"Count of {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plt.figure()
sns.countplot(x="attack_detected", data=df)
plt.title("Target Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(18, 12))

for i, col in enumerate(cat_cols, 1):
    plt.subplot(2, 3, i)
    counts = df[col].value_counts() 
    plt.pie(counts.values,labels=counts.index,autopct='%1.1f%%',startangle=90)
    plt.title(f"{col} Distribution")
plt.tight_layout()
plt.show()

## Bivariate Analysis 
**Target vs Numerical Features**

In [ ]:
for col in num_cols:
    if col != "attack_detected":
        plt.figure()
        sns.boxplot(x="attack_detected", y=col, data=df)
        plt.title(f"{col} vs Attack")
        plt.show()

In [ ]:
for col in num_cols:
    if col != "attack_detected":
        group0 = df[df["attack_detected"] == 0][col]
        group1 = df[df["attack_detected"] == 1][col]
        stat, p = ttest_ind(group0, group1)
        print(f"{col} | p-value: {p}")

**Target vs Categorical Features** 

In [ ]:
for col in cat_cols:
    plt.figure()
    sns.countplot(x=col, hue="attack_detected", data=df)
    plt.title(f"{col} vs Attack")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for col in cat_cols:
    print(f"\n{col}")
    print(pd.crosstab(df[col], df["attack_detected"], normalize="index"))

## MultiVariate Analysis 
**Correlation Matrix**

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
selected_features = [col for col in num_cols if col != "attack_detected"][:4]
selected_features.append("attack_detected")

sns.pairplot(df[selected_features], hue="attack_detected")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

# Encode categorical features
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])

X = df_encoded.drop("attack_detected", axis=1)
y = df_encoded["attack_detected"]

model = RandomForestClassifier(random_state=42)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind="barh")
plt.title("Feature Importance (Baseline Model)")
plt.show()